

# DOWNLOAD



In [ ]:
!pip install transformers datasets evaluate sentencepiece accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


# 1_LOAD DATASET

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from datasets import load_from_disk

AOPE_path = '/content/drive/MyDrive/CS221.Q11/DATASET/EduRABSA_AOPE'
ASTE_path = '/content/drive/MyDrive/CS221.Q11/DATASET/EduRABSA_ASTE'

try:
    # Load dataset từ thư mục trên Drive
    AOPE_dataset = load_from_disk(AOPE_path)

    print("Đã load dataset thành công!")
    print(AOPE_dataset)

    # Kiểm tra thử 1 mẫu dữ liệu trong tập train
    print("\nVí dụ dữ liệu:")
    print(AOPE_dataset['train'][0])

except FileNotFoundError:
    print(f"Không tìm thấy thư mục tại: {AOPE_path}")
    print("Bạn vui lòng kiểm tra lại đường dẫn xem đã đúng chưa nhé.")



## CHIA LẠI TRAIN/VAL/TEST

In [ ]:
from datasets import load_from_disk, concatenate_datasets, DatasetDict

# 1. Load dữ liệu từ Drive (Giả sử bạn chọn task ASTE vì nó bao hàm cả Sentiment)
# Nếu bạn muốn làm AOPE thì thay đường dẫn tương ứng, logic vẫn y hệt.
dataset_path = '/content/drive/MyDrive/CS221.Q11/DATASET/EduRABSA_ASTE'
dataset = load_from_disk(dataset_path)

print(f"Cấu trúc gốc: {dataset}")

# 2. Gộp Train và Test cũ lại thành 1 cục to
full_data = concatenate_datasets([dataset['train'], dataset['test']])

# 3. Chia lại theo tỷ lệ 80% Train, 10% Val, 10% Test
# Đầu tiên tách 80% Train và 20% còn lại
train_testvalid = full_data.train_test_split(test_size=0.2, seed=42)

# Tách 20% còn lại thành 50% Val và 50% Test (tức là 10% tổng mỗi cái)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Gom lại thành DatasetDict hoàn chỉnh
new_dataset = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

print("Cấu trúc mới sau khi chia lại (80/10/10):")
print(new_dataset)

# BERT

## 2_PREPROCESSING

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_from_disk, Dataset, DatasetDict
from transformers import AutoTokenizer

# 1. Load Dataset (Đảm bảo bạn đã mount drive và đường dẫn đúng)
dataset_path = '/content/drive/MyDrive/CS221.Q11/DATASET/EduRABSA_ASTE'
# dataset = load_from_disk(dataset_path)
# Lưu ý: Nếu bạn đã chia lại split ở bước trước thì dùng biến 'new_dataset', nếu không thì load lại
# Để demo mình giả định biến dataset đã sẵn sàng
print(dataset)

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# --- HÀM TẠO NHÃN BIO (CHO ASPECT VÀ OPINION) ---
def create_bio_tags(examples, target_col='aspect'):
    # target_col: 'aspect' hoặc 'opinion'
    # Output: 'ner_tags' dạng số [0, 1, 2, 0...] (O, B, I)

    batch_tokens = []
    batch_tags = []

    for idx, sentence in enumerate(examples['sentence']):
        tokens = sentence.split() # Tokenize đơn giản bằng dấu cách
        tags = [0] * len(tokens) # 0 là 'O'

        # Lấy danh sách triplets của câu này
        triplets = examples['triplets'][idx]

        for triplet in triplets:
            # Lấy text của aspect hoặc opinion
            # Cấu trúc triplet có thể là dict hoặc list tùy lúc load
            target_text = triplet[target_col] if isinstance(triplet, dict) else triplet[0 if target_col=='aspect' else 1]

            # --- ĐIỂM YẾU CỦA BASELINE ---
            # Nếu là 'null' (Implicit), bỏ qua luôn -> Mô hình không học được cái này
            if target_text is None or str(target_text).lower() == 'null':
                continue

            target_tokens = str(target_text).split()
            len_target = len(target_tokens)

            # Tìm vị trí (Exact Match)
            for i in range(len(tokens) - len_target + 1):
                if tokens[i:i+len_target] == target_tokens:
                    tags[i] = 1 # B (Begin)
                    for j in range(1, len_target):
                        tags[i+j] = 2 # I (Inside)

        batch_tokens.append(tokens)
        batch_tags.append(tags)

    return {'tokens': batch_tokens, 'ner_tags': batch_tags}

# --- HÀM TẠO DATA CHO SENTIMENT (CLASSIFICATION) ---
def create_sentiment_data(examples):
    # Input: "[CLS] Câu review [SEP] Aspect [SEP] Opinion"
    # Label: Pos/Neg/Neu
    input_texts = []
    labels = []
    label_map = {'POSITIVE': 0, 'NEGATIVE': 1, 'NEUTRAL': 2}

    for idx, sentence in enumerate(examples['sentence']):
        triplets = examples['triplets'][idx]
        for triplet in triplets:
            asp = triplet['aspect'] if isinstance(triplet, dict) else triplet[0]
            opi = triplet['opinion'] if isinstance(triplet, dict) else triplet[1]
            sen = triplet['sentiment'] if isinstance(triplet, dict) else triplet[2]

            # Bỏ qua implicit
            if str(asp).lower() == 'null' or str(opi).lower() == 'null': continue

            # Tạo input chuẩn cho BERT classification
            text = f"{sentence} [SEP] {asp} [SEP] {opi}"

            if sen.upper() in label_map:
                input_texts.append(text)
                labels.append(label_map[sen.upper()])

    return {'text': input_texts, 'label': labels}

# Tạo 3 bộ dataset riêng biệt
print("Đang tạo dataset cho Aspect Extraction...")
ae_dataset = dataset.map(lambda x: create_bio_tags(x, 'aspect'), batched=True, remove_columns=dataset['train'].column_names)

print("Đang tạo dataset cho Opinion Extraction...")
oe_dataset = dataset.map(lambda x: create_bio_tags(x, 'opinion'), batched=True, remove_columns=dataset['train'].column_names)

print("Đang tạo dataset cho Sentiment Analysis...")
sa_dataset = dataset.map(create_sentiment_data, batched=True, remove_columns=dataset['train'].column_names)

##3_TRAINING

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

# Metric đánh giá token (SeqEval)
seqeval = evaluate.load("seqeval")
label_list = ["O", "B", "I"]

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

def tokenize_and_align(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None: label_ids.append(-100)
            elif word_idx != previous_word_idx: label_ids.append(label[word_idx])
            else: label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs



In [ ]:
# --- 1. TRAIN ASPECT EXTRACTOR ---
print("\n Training Aspect Model...")
tokenized_ae = ae_dataset.map(tokenize_and_align, batched=True)
model_ae = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=3)

args = TrainingArguments(
    "bert-aspect-v1",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3, # Tăng lên 5-10 nếu cần kết quả tốt hơn
    weight_decay=0.01
)
data_collator = DataCollatorForTokenClassification(tokenizer)
trainer_ae = Trainer(model_ae, args, train_dataset=tokenized_ae["train"], eval_dataset=tokenized_ae["test"], tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
trainer_ae.train()
trainer_ae.save_model("my_aspect_model") # Lưu model



In [ ]:
# --- 2. TRAIN OPINION EXTRACTOR ---
print("\n Training Opinion Model...")
tokenized_oe = oe_dataset.map(tokenize_and_align, batched=True)
model_oe = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=3)

args_oe = TrainingArguments("bert-opinion-v1", evaluation_strategy="epoch", learning_rate=2e-5, per_device_train_batch_size=16, num_train_epochs=3)
trainer_oe = Trainer(model_oe, args_oe, train_dataset=tokenized_oe["train"], eval_dataset=tokenized_oe["test"], tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
trainer_oe.train()
trainer_oe.save_model("my_opinion_model") # Lưu model



In [ ]:
# --- 3. TRAIN SENTIMENT CLASSIFIER ---
print("\n Training Sentiment Model...")
from transformers import AutoModelForSequenceClassification

def preprocess_sa(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=128)

tokenized_sa = sa_dataset.map(preprocess_sa, batched=True)
model_sa = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

args_sa = TrainingArguments("bert-sentiment-v1", evaluation_strategy="epoch", learning_rate=2e-5, per_device_train_batch_size=16, num_train_epochs=3)
trainer_sa = Trainer(model_sa, args_sa, train_dataset=tokenized_sa["train"], eval_dataset=tokenized_sa["test"], tokenizer=tokenizer)
trainer_sa.train()
trainer_sa.save_model("my_sentiment_model") # Lưu model

##4_INFERENCE

In [ ]:
import torch

# Load lại 3 model đã train
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ae = AutoModelForTokenClassification.from_pretrained("my_aspect_model").to(device)
model_oe = AutoModelForTokenClassification.from_pretrained("my_opinion_model").to(device)
model_sa = AutoModelForSequenceClassification.from_pretrained("my_sentiment_model").to(device)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

label_map_inv = {0: 'POSITIVE', 1: 'NEGATIVE', 2: 'NEUTRAL'}

def extract_spans_from_bio(text, model):
    # Hàm helper: Chuyển logits -> BIO tags -> List các cụm từ
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    spans = []
    current_span = []

    # Logic ghép từ BIO: Nếu gặp B thì bắt đầu span mới, I thì nối tiếp
    for token, label_id in zip(tokens, predictions):
        if token in ['[CLS]', '[SEP]', '[PAD]']: continue
        if token.startswith("##"): token = token[2:] # Xử lý subword của BERT

        if label_id == 1: # B
            if current_span: spans.append(" ".join(current_span))
            current_span = [token]
        elif label_id == 2: # I
            if current_span: current_span.append(token)
        else: # O
            if current_span:
                spans.append(" ".join(current_span))
                current_span = []
    if current_span: spans.append(" ".join(current_span))
    return spans

def pipeline_inference(sentence):
    # 1. Trích xuất Aspects
    aspects = extract_spans_from_bio(sentence, model_ae)

    # 2. Trích xuất Opinions
    opinions = extract_spans_from_bio(sentence, model_oe)

    results = []

    # Nếu không tìm thấy gì thì trả về rỗng
    if not aspects or not opinions:
        return []

    # 3. HEURISTIC PAIRING (ĐIỂM YẾU):
    # Ghép tất cả Aspect tìm được với tất cả Opinion tìm được (Cartesian Product)
    # Vì Pipeline không biết cái nào đi với cái nào
    for asp in aspects:
        for opi in opinions:
            # 4. Phân loại cảm xúc cho cặp
            text_pair = f"{sentence} [SEP] {asp} [SEP] {opi}"
            inputs = tokenizer(text_pair, return_tensors="pt", truncation=True, max_length=128).to(device)

            with torch.no_grad():
                outputs = model_sa(**inputs)

            pred_idx = torch.argmax(outputs.logits).item()
            sentiment = label_map_inv[pred_idx]

            # Format output giống dataset: (aspect, opinion, sentiment)
            results.append((asp, opi, sentiment))

    return results

# --- TEST THỬ ---
sample_text = "The teacher is knowledgeable but the assignments are hard."
print(f"Câu review: {sample_text}")
pred_triplets = pipeline_inference(sample_text)
print(f"Pipeline dự đoán: {pred_triplets}")

# Ví dụ về việc thất bại với Implicit
implicit_text = "Too expensive!"
print(f"\nCâu Implicit: {implicit_text}")
print(f"Pipeline dự đoán: {pipeline_inference(implicit_text)}")
# Kết quả dự kiến sẽ là [] (Rỗng) vì không tìm thấy Aspect "Cost/Price" trong câu.

# BiLSTM-CRF

In [ ]:
!pip install pytorch-crf datasets

## LOAD DATASET

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from datasets import load_from_disk
import numpy as np
from collections import Counter

# 1. Load Dataset
dataset_path = '/content/drive/MyDrive/CS221.Q11/DATASET/EduRABSA_ASTE'
dataset = load_from_disk(dataset_path)

# 2. Xây dựng bộ từ điển (Vocabulary)
# BiLSTM không dùng Tokenizer xịn như BERT, ta phải tự build vocab
print("Đang xây dựng Vocabulary...")
word_counts = Counter()
for sentence in dataset['train']['sentence']:
    word_counts.update(sentence.lower().split())

# Chỉ giữ lại từ xuất hiện > 1 lần để giảm nhiễu (tùy chọn)
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counts.items():
    if count > 1:
        vocab[word] = len(vocab)

print(f"Kích thước bộ từ điển: {len(vocab)}")

# 3. Hàm chuyển dữ liệu sang BIO Tags (giống pipeline BERT)
tag2idx = {"O": 0, "B": 1, "I": 2}
idx2tag = {0: "O", 1: "B", 2: "I"}

def create_bio_data_indices(examples, target_col='aspect'):
    input_ids_list = []
    tags_list = []

    for idx, sentence in enumerate(examples['sentence']):
        tokens = sentence.lower().split()

        # Chuyển token sang ID
        input_ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
        tags = [0] * len(tokens)

        triplets = examples['triplets'][idx]
        for triplet in triplets:
            target_text = triplet[target_col] if isinstance(triplet, dict) else triplet[0 if target_col=='aspect' else 1]

            # --- BỎ QUA IMPLICIT (Điểm yếu chí mạng) ---
            if target_text is None or str(target_text).lower() == 'null':
                continue

            target_tokens = str(target_text).lower().split()
            len_target = len(target_tokens)

            for i in range(len(tokens) - len_target + 1):
                if tokens[i:i+len_target] == target_tokens:
                    tags[i] = 1 # B
                    for j in range(1, len_target):
                        tags[i+j] = 2 # I

        input_ids_list.append(input_ids)
        tags_list.append(tags)

    return {'input_ids': input_ids_list, 'labels': tags_list}

# Tạo data cho Aspect và Opinion riêng biệt
print("Processing data...")
ae_data = dataset.map(lambda x: create_bio_data_indices(x, 'aspect'), batched=True, remove_columns=dataset['train'].column_names)
oe_data = dataset.map(lambda x: create_bio_data_indices(x, 'opinion'), batched=True, remove_columns=dataset['train'].column_names)

## EMBEDINGS


In [ ]:
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tag_to_ix, embedding_dim, hidden_dim):
        super(BiLSTM_CRF, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.tag_to_ix = tag_to_ix
        self.tagset_size = len(tag_to_ix)

        # 1. Embedding Layer
        self.word_embeds = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. LSTM Layer (Bidirectional)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)

        # 3. Linear Layer (Project về không gian nhãn)
        self.hidden2tag = nn.Linear(hidden_dim, self.tagset_size)

        # 4. CRF Layer
        self.crf = CRF(self.tagset_size, batch_first=True)

    def forward(self, input_ids, tags=None, mask=None):
        embeds = self.word_embeds(input_ids)
        lstm_out, _ = self.lstm(embeds)
        emissions = self.hidden2tag(lstm_out)

        if tags is not None:
            # Training: Trả về Loss (Negative Log Likelihood)
            # mask giúp CRF bỏ qua các token padding
            log_likelihood = self.crf(emissions, tags, mask=mask, reduction='mean')
            return -log_likelihood
        else:
            # Inference: Trả về đường đi tốt nhất (Viterbi Decoding)
            return self.crf.decode(emissions, mask=mask)

## TRAINING


In [ ]:
# Cấu hình
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
BATCH_SIZE = 32
LEARNING_RATE = 0.01 # LSTM thường cần LR cao hơn BERT
EPOCHS = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hàm padding custom
def collate_fn(batch):
    input_ids = [torch.tensor(item['input_ids']) for item in batch]
    labels = [torch.tensor(item['labels']) for item in batch]

    # Pad sequences
    input_ids_pad = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    labels_pad = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=0)

    # Tạo mask (1 cho token thật, 0 cho padding)
    mask = (input_ids_pad != 0).byte()

    return input_ids_pad, labels_pad, mask

# Hàm train chung
def train_model(train_dataset, model_name="model"):
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    model = BiLSTM_CRF(len(vocab), tag2idx, EMBEDDING_DIM, HIDDEN_DIM).to(device)
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4) # SGD thường ổn định hơn cho LSTM thuần

    print(f"--- Bắt đầu train {model_name} ---")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for input_ids, labels, mask in train_loader:
            input_ids, labels, mask = input_ids.to(device), labels.to(device), mask.to(device)

            model.zero_grad()
            loss = model(input_ids, labels, mask)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

    return model

# Train 2 model riêng biệt
model_ae_bilstm = train_model(ae_data['train'], "Aspect Extractor (BiLSTM)")
model_oe_bilstm = train_model(oe_data['train'], "Opinion Extractor (BiLSTM)")

## INFERENCE

In [ ]:
def extract_bilstm(sentence, model):
    model.eval()
    tokens = sentence.lower().split()
    input_ids = torch.tensor([[vocab.get(t, vocab["<UNK>"]) for t in tokens]]).to(device)
    mask = torch.tensor([[1] * len(tokens)], dtype=torch.uint8).to(device)

    with torch.no_grad():
        # Decode ra list index [0, 1, 2...]
        prediction_indices = model(input_ids, mask=mask)[0]

    # Logic chuyển Index -> Text Span
    spans = []
    current_span = []
    for token, idx in zip(tokens, prediction_indices):
        tag = idx2tag[idx]
        if tag == "B":
            if current_span: spans.append(" ".join(current_span))
            current_span = [token]
        elif tag == "I":
            if current_span: current_span.append(token)
        else: # O
            if current_span:
                spans.append(" ".join(current_span))
                current_span = []
    if current_span: spans.append(" ".join(current_span))

    return spans

# Test thử
test_sentence = "The teacher is very kind but the exams are difficult"
print(f"Câu: {test_sentence}")

aspects = extract_bilstm(test_sentence, model_ae_bilstm)
opinions = extract_bilstm(test_sentence, model_oe_bilstm)

print(f"BiLSTM Aspects: {aspects}")
print(f"BiLSTM Opinions: {opinions}")

# --- KẾT QUẢ DỰ KIẾN CHO ĐỒ ÁN ---
# 1. Câu Implicit: "It's a waste of money"
# -> BiLSTM Aspects: [] (Không rút được 'Course' hay 'Tuition' vì không có từ đó)
# -> Đây là bằng chứng cho việc model cổ điển thất bại.

# T5-BASE

In [ ]:
!pip install transformers datasets evaluate sentencepiece accelerate -q

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_from_disk
import numpy as np

# 1. Load Dataset
dataset_path = '/content/drive/MyDrive/CS221.Q11/DATASET/EduRABSA_ASTE'
dataset = load_from_disk(dataset_path)

# 2. Load Model & Tokenizer
checkpoint = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(checkpoint)
model = T5ForConditionalGeneration.from_pretrained(checkpoint)

## PREPROCESSING

In [ ]:
def preprocess_function(examples):
    inputs = []
    targets = []

    # Prefix giúp T5 hiểu task (tùy chọn, nhưng tốt cho t5-base)
    prefix = "aste: "

    for i in range(len(examples['sentence'])):
        review = examples['sentence'][i]
        triplets = examples['triplets'][i]

        # 1. Input: Thêm prefix
        inputs.append(prefix + review)

        # 2. Target: Tạo chuỗi (aspect, opinion, sentiment)
        formatted_triplets = []
        for t in triplets:
            # Lấy data (xử lý tùy xem t là dict hay list)
            asp = t['aspect'] if isinstance(t, dict) else t[0]
            opi = t['opinion'] if isinstance(t, dict) else t[1]
            sen = t['sentiment'] if isinstance(t, dict) else t[2]

            # --- KHÁC BIỆT SO VỚI BASELINE ---
            # Ta KHÔNG dùng lệnh continue/bỏ qua nếu là null.
            # Ta chuyển None/null thành chuỗi "null" để model học.
            if asp is None or str(asp).lower() == 'null':
                asp = "null"
            if opi is None or str(opi).lower() == 'null':
                opi = "null"

            # Format: (aspect, opinion, sentiment)
            formatted_triplets.append(f"({asp}, {opi}, {sen})")

        # Nối các bộ ba bằng dấu chấm phẩy
        target_str = "; ".join(formatted_triplets)
        targets.append(target_str)

    # Tokenize Input và Target
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    # Thay thế padding token id bằng -100 để không tính loss cho padding
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Áp dụng xử lý
print("Đang xử lý dữ liệu cho T5...")
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset['train'].column_names)

# Kiểm tra thử 1 mẫu
print("\n--- Mẫu dữ liệu sau khi xử lý ---")
print("Input:", tokenizer.decode(tokenized_datasets['train'][0]['input_ids'], skip_special_tokens=True))
print("Target:", tokenizer.decode([l for l in tokenized_datasets['train'][0]['labels'] if l != -100], skip_special_tokens=True))

## TRAINING

In [ ]:
# Cấu hình huấn luyện
training_args = Seq2SeqTrainingArguments(
    output_dir="./edurabsa_t5_results",
    evaluation_strategy="epoch",
    learning_rate=3e-4, # T5 thường thích LR cao hơn BERT chút (1e-4 đến 3e-4)
    per_device_train_batch_size=8, # T5-base khá nặng, giảm batch nếu OOM
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True, # Quan trọng: Để model sinh text khi eval
    fp16=True, # Dùng GPU tối ưu
    save_strategy="epoch",
    load_best_model_at_end=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"], # Hoặc validation nếu bạn đã chia
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(" Bắt đầu Fine-tuning T5...")
trainer.train()

# Lưu model
trainer.save_model("my_t5_aste_model")

## INFERENCES

In [ ]:
def predict_t5(sentence, model_path="my_t5_aste_model"):
    # Load model đã train
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = T5Tokenizer.from_pretrained("t5-base") # Tokenizer gốc
    model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)

    # Chuẩn bị input
    input_text = "aste: " + sentence
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    # Sinh văn bản (Generate)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=128,
        num_beams=4, # Beam search giúp câu sinh ra chuẩn hơn
        early_stopping=True
    )

    # Decode ra text
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return prediction

print("\n--- KẾT QUẢ SO SÁNH (Dùng cho Báo cáo) ---")

# 1. Trường hợp Explicit (Dễ)
text_easy = "The professor is knowledgeable but the lectures are boring."
print(f"Input: {text_easy}")
print(f"T5 Output: {predict_t5(text_easy)}")
# Kỳ vọng: (professor, knowledgeable, positive); (lectures, boring, negative)

# 2. Trường hợp Implicit (Khó - Pipeline/BiLSTM sẽ thất bại)
text_hard = "Too expensive!"
print(f"\nInput: {text_hard}")
print(f"T5 Output: {predict_t5(text_hard)}")
# Kỳ vọng: (null, expensive, negative) hoặc (price, expensive, negative)
# -> Đây là bằng chứng T5 vượt trội.